# To AI or Not to AI: measuring what "familiar" looks like

This notebook **does not train anything**. It loads the model trained by the main notebook (attach that notebook's output as an input), and measures what familiar images look like inside it:

1. Takes images from the original training, validation, test and held-out sets (from the main run's `manifest.csv`).
2. Records the model's 576 internal features for each one.
3. Fits the familiarity check on **training** images, and sets its threshold on **validation** images only.
4. Reports what the check does on the test set and on the **held-out** sets, which are never used to choose anything.
5. Re-exports the ONNX model with a second output (the features) and packages `familiarity.npz`.

**On Kaggle:** use *Copy & Edit* on the main notebook (so the datasets stay attached), import this notebook into the copy, add the main notebook's latest output as an input, set a GPU, turn internet on, then *Save & Run All*. Takes about 20 minutes.

> **Note:** this notebook clones the commit in `REPO_REF`, so its imports use that commit's file names.
> On newer commits the engine lives in `src.ml.predictor` and `src.ml.familiarity`,
> and the training dependencies are in `requirements/train.txt`. Update the imports if you change `REPO_REF`.

In [ ]:
# Setup: clone the repository at the commit with the familiarity code
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gismal/to-ai-or-not-to-ai.git"
REPO_REF = "714ff273fd3cc95443630b402239eefebd9724a6"   # the commit that adds src/familiarity.py

%cd /kaggle/working
!rm -rf repo && git clone --quiet {REPO_URL} repo && cd repo && git checkout --quiet {REPO_REF}
%cd /kaggle/working/repo
!pip install -q onnx onnxruntime onnxscript pydantic-settings python-dotenv

GIT_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
print("Code at commit", GIT_COMMIT)

# src.config needs these values to import. They are placeholders, not secrets.
Path(".env").write_text(
    "API_KEY=training-only\n"
    "DATABASE_URL=sqlite+aiosqlite:///unused.db\n"
    "MODEL_THRESHOLD=0.75\n"
    "GRAY_AREA_MARGIN=0.35\n"
)

In [ ]:
# Configuration
SEED = 42
TRAIN_PER_CLASS = 4_000      # training images per class used to describe "familiar"
PERCENTILE = 95.0            # decided BEFORE looking at any held-out result: flag ~5% of familiar images
SENSITIVITY = (90.0, 99.0)   # shown for information only; not used to choose anything
MODEL_VERSION = "v2.1.0"

In [ ]:
# The main run's output: model checkpoint, metadata and the manifest of which image went where
import json
import zipfile

import pandas as pd

zips = sorted(Path("/kaggle/input").rglob("aon_model.zip"))
assert zips, "Attach the main notebook's output (the version that produced aon_model.zip) as an input."
RUN_DIR = Path("/kaggle/working/main_run")
with zipfile.ZipFile(zips[-1]) as z:
    z.extractall(RUN_DIR)
print("Using", zips[-1])

meta = json.loads((RUN_DIR / "models/metadata.json").read_text())
manifest = pd.read_csv(RUN_DIR / "reports/manifest.csv")
manifest["holdout"] = manifest.holdout.astype(str).str.lower() == "true"
missing = (~manifest.original.map(lambda p: Path(p).exists())).sum()
print(f"{len(manifest):,} images in the manifest; {missing:,} original files not found")
assert len(manifest) > 50_000, "This is the small test run's output, not the full run: attach the right version."
assert missing < 0.01 * len(manifest), "Most originals are missing: are the same datasets attached?"

In [ ]:
# Features and scores for a sample of every split, preprocessed exactly as the API does it
import io
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch

from scripts.training.familiarity import WithFeatures
from scripts.training.model import build_model
from src.inference import _preprocess

device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_model(2, freeze_backbone=False, pretrained=False)
checkpoint = torch.load(RUN_DIR / "models/best_checkpoint.pt", map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
net = WithFeatures(model).to(device).eval()

train = manifest[manifest.split == "train"].groupby("label", group_keys=False).apply(
    lambda g: g.sample(min(len(g), TRAIN_PER_CLASS), random_state=SEED))
sample = pd.concat([train, manifest[manifest.split != "train"]]).reset_index(drop=True)


def load(path):
    try:
        _, tensor, error = _preprocess(io.BytesIO(Path(path).read_bytes()))
        return None if error is not None else tensor[0]
    except Exception:
        return None


features, p_ai, keep = [], [], []
with ThreadPoolExecutor(8) as pool:
    for start in range(0, len(sample), 256):
        tensors = list(pool.map(load, sample.original[start:start + 256]))
        ok = [i for i, t in enumerate(tensors) if t is not None]
        x = torch.from_numpy(np.stack([tensors[i] for i in ok])).to(device)
        with torch.no_grad():
            logits, f = net(x)
        features.append(f.cpu().numpy())
        p_ai.append(torch.softmax(logits, dim=1)[:, 0].cpu().numpy())   # class 0 is AI_GENERATED
        keep += [start + i for i in ok]
        if start % 5120 == 0:
            print(f"  {start:,} / {len(sample):,}")

sample = sample.loc[keep].reset_index(drop=True)
sample["p_ai"] = np.concatenate(p_ai)
sample["is_ai"] = sample.label == "AI_GENERATED"
X = np.concatenate(features)
print(f"Features for {len(sample):,} images, {X.shape[1]} numbers each")

In [ ]:
# Fit on training images, choose the threshold on validation images, then look at everything else
from scripts.training.familiarity import fit
from src.familiarity import mahalanobis

upper = meta["evaluation"]["thresholds"]["ai"]       # the thresholds in production
lower = meta["evaluation"]["thresholds"]["real"]
part = {name: sample.index[mask].to_numpy() for name, mask in {
    "train": sample.split == "train", "val": sample.split == "val",
    "test": (sample.split == "test") & ~sample.holdout, "held-out": sample.holdout}.items()}
y_train = sample.loc[part["train"], "is_ai"].astype(int).to_numpy()


def summarize(rows, unfamiliar=None):
    df = sample.loc[rows]
    says_ai, says_real = df.p_ai >= upper, df.p_ai <= lower
    decided = (says_ai | says_real).to_numpy()
    if unfamiliar is not None:
        decided = decided & ~unfamiliar
    right = ((says_ai & df.is_ai) | (says_real & ~df.is_ai)).to_numpy()
    return {"images": len(df),
            "flagged_unfamiliar": round(float(unfamiliar.mean()), 4) if unfamiliar is not None else 0.0,
            "decided_share": round(float(decided.mean()), 4),
            "accuracy_when_decided": round(float(right[decided].mean()), 4) if decided.any() else None}


results, stats_by_p = {}, {}
for p in (PERCENTILE, *SENSITIVITY):
    stats = fit(X[part["train"]], y_train, X[part["val"]], percentile=p)
    stats_by_p[p] = stats
    for name in ("test", "held-out"):
        d = mahalanobis(X[part[name]], stats["means"], stats["precision"])
        results[(p, name)] = summarize(part[name], d > stats["threshold"])
before = {name: summarize(part[name]) for name in ("test", "held-out")}

table = pd.DataFrame({**{("no check", n): r for n, r in before.items()},
                      **{(f"{p:g}th pct", n): r for (p, n), r in results.items()}}).T
print(table.to_string())
print(f"\nChosen in advance: the {PERCENTILE:g}th percentile. The other rows are for information only.")

In [ ]:
# Package: the two-output ONNX model, the familiarity statistics, and updated metadata
import hashlib
import shutil

import onnxruntime as ort

from scripts.training.export import export_onnx

OUT = Path("/kaggle/working/models")
OUT.mkdir(exist_ok=True)
export_onnx(model.cpu().eval(), OUT / "model_v1.onnx")
shutil.copy(RUN_DIR / "models/best_checkpoint.pt", OUT / "best_checkpoint.pt")
chosen = stats_by_p[PERCENTILE]
np.savez(OUT / "familiarity.npz", **chosen)

# The new ONNX file must give the same scores and features as PyTorch
session = ort.InferenceSession(str(OUT / "model_v1.onnx"), providers=["CPUExecutionProvider"])
x = torch.from_numpy(np.stack([load(p) for p in sample.original[:8]]))
onnx_logits, onnx_features = session.run(None, {"input": x.numpy()})
with torch.no_grad():
    want_logits, want_features = WithFeatures(model.cpu())(x)
assert np.allclose(onnx_logits, want_logits.numpy(), atol=1e-3)
assert np.allclose(onnx_features, want_features.numpy(), atol=1e-3)
print("ONNX scores and features match PyTorch.")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


meta["model_version"] = MODEL_VERSION
meta["familiarity"] = {
    "method": "Mahalanobis distance to the nearest class centre, on the 576 pooled features",
    "fitted_on": f"{len(part['train']):,} training images", "threshold_chosen_on": "validation set",
    "percentile": PERCENTILE, "threshold": round(float(chosen["threshold"]), 4),
    "test_set": results[(PERCENTILE, "test")], "held_out": results[(PERCENTILE, "held-out")],
    "without_check": before, "measured_at_commit": GIT_COMMIT,
}
files = [f.name for f in sorted(OUT.iterdir()) if f.suffix != ".json"]
meta["sha256"] = {name: sha256(OUT / name) for name in files}
(OUT / "metadata.json").write_text(json.dumps(meta, indent=4))

out = Path("/kaggle/working/aon_model_v2_1.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        z.write(f, f"models/{f.name}")
shutil.rmtree(RUN_DIR, ignore_errors=True)
print(f"{out} ({out.stat().st_size / 1e6:.1f} MB): download it from the Output tab.")
print(json.dumps(meta["familiarity"], indent=2))